In [1]:
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier, LGBMRegressor
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV

In [2]:
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
df_train = pd.read_csv("data/train_final.csv")

In [4]:
df_train.head()

,cust_id,sale_id_nunique,prod_id_count,sale_revenue_sum,sale_revenue_mean,sale_discount_applied_sum,is_returned_mean,is_returned_last,pack_delay_days_max,is_jan_july_mean,prod_web_only_mean,prod_brand_mode,prod_size_nunique,recency_days,discount_affinity,recent_spend_ratio,spend_trajectory_ratio,churn_risk_ratio,revenue_2018_2019
0,222agnowc53dykbq,1,1,89.95,89.950000,0.00,0.000000,0,0,0.0,0.0,geox,1,383,0.000000,0.0,0.0,383.0,0.0
1,222ny4m63rmalpdl,1,3,125.93,41.976667,-45.16,0.333333,0,0,0.0,0.0,other,3,374,-0.358612,0.0,0.0,37400.0,0.0
2,223jend5smd4ptmc,1,1,89.95,89.950000,0.00,0.000000,0,0,0.0,0.0,tommy hilfiger,1,488,0.000000,0.0,0.0,488.0,0.0
3,223xvc4rbjatlnev,1,2,116.14,58.070000,-49.76,0.000000,0,7,1.0,0.0,teva,2,520,-0.428448,0.0,0.0,52000.0,0.0
4,223y2r357elerbis,1,2,47.97,23.985000,-58.96,0.500000,0,0,1.0,0.0,tamaris,1,348,-1.229102,0.0,479700.0,34800.0,0.0


In [5]:
TARGET_COLUMN = "revenue_2018_2019"

In [6]:
X_train = df_train.drop(columns=["cust_id", TARGET_COLUMN])

In [7]:
y_train = df_train[TARGET_COLUMN]

In [8]:
df_valid = pd.read_csv("data/valid_final.csv")

In [9]:
X_valid = df_valid.drop(columns=["cust_id", TARGET_COLUMN])

In [10]:
y_valid = df_valid[TARGET_COLUMN]

In [11]:
categorical_cols = ["prod_brand_mode"]

In [12]:
for col in categorical_cols:
    X_train[col] = X_train[col].astype("category")
    X_valid[col] = X_valid[col].astype("category")

In [13]:
y_train_binary = (y_train > 0).astype(int)

In [14]:
y_valid_binary = (y_valid > 0).astype(int)

In [15]:
y_train_binary.value_counts()

revenue_2018_2019
0    59147
1    34141
Name: count, dtype: int64

## Training the classifier

In [16]:
clf = LGBMClassifier(
    n_estimators=500, learning_rate=0.03, num_leaves=31, random_state=42
)

In [17]:
clf.fit(X_train, y_train_binary, categorical_feature=["prod_brand_mode"])

[LightGBM] [Info] Number of positive: 34141, number of negative: 59147
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002756 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2548
[LightGBM] [Info] Number of data points in the train set: 93288, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.365974 -> initscore=-0.549527
[LightGBM] [Info] Start training from score -0.549527


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.03
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [18]:
p_valid = clf.predict_proba(X_valid)[:, 1]  # type: ignore

In [19]:
p_valid

array([0.25254293, 0.16114102, 0.20626025, ..., 0.7952165 , 0.25343424,
       0.27916581], shape=(23303,))

## Training the regressor

In [20]:
mask = y_train > 0

In [21]:
X_train_reg = X_train[mask].copy()

In [22]:
y_train_reg = y_train[mask].copy()

In [23]:
upper_cap = np.percentile(y_train_reg, 99)  # clipping extreme outliers
y_train_reg = np.clip(y_train_reg, 0, upper_cap)

In [24]:
reg = LGBMRegressor(
    n_estimators=800, learning_rate=0.03, num_leaves=31, random_state=42
)

In [25]:
reg.fit(X_train_reg, y_train_reg, categorical_feature=["prod_brand_mode"])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000969 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2529
[LightGBM] [Info] Number of data points in the train set: 34141, number of used features: 17
[LightGBM] [Info] Start training from score 190.894691


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.03
,n_estimators,800
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [26]:
amount_valid = reg.predict(X_valid)

In [27]:
amount_valid = np.clip(amount_valid, 0, None)  # type: ignore

## Combining the two models

In [28]:
p_valid_adj = np.clip(p_valid, 0.05, 0.95)

In [29]:
p_valid_adj

array([0.25254293, 0.16114102, 0.20626025, ..., 0.7952165 , 0.25343424,
       0.27916581], shape=(23303,))

In [30]:
y_pred = p_valid_adj * amount_valid

In [31]:
mae = mean_absolute_error(y_valid, y_pred)

In [32]:
spearman_corr, p_value = spearmanr(y_valid, y_pred)

In [33]:
print("-" * 30)
print(f"Validation MAE: {mae:.2f}")
print(f"Validation Spearman Correlation: {spearman_corr:.4f}")
print("-" * 30)

------------------------------
Validation MAE: 77.62
Validation Spearman Correlation: 0.3818
------------------------------


so the two step models is currently underperforming compared to the lightgbm. running some diagnostics to find the issue

In [34]:
auc = roc_auc_score(y_valid_binary, p_valid)

In [35]:
print("AUC:", auc)

AUC: 0.7124978129132526


In [36]:
clf = LGBMClassifier(random_state=10, n_jobs=-1)

In [37]:
param_grid_clf = {
    "n_estimators": [300, 500, 800],
    "learning_rate": [0.01, 0.03, 0.05],
    "num_leaves": [31, 64, 128],
    "min_child_samples": [20, 50, 100],
}

In [38]:
clf_search = RandomizedSearchCV(
    clf,
    param_grid_clf,
    n_iter=10,
    scoring="roc_auc",
    cv=3,
    n_jobs=1,
    verbose=2,
    random_state=17,
)

In [39]:
print("Starting the hyperparameter tuning for the classifier ...")
clf_search.fit(X_train, y_train_binary)

Starting the hyperparameter tuning for the classifier ...
Fitting 3 folds for each of 10 candidates, totalling 30 fits
[LightGBM] [Info] Number of positive: 22761, number of negative: 39431
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001275 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2482
[LightGBM] [Info] Number of data points in the train set: 62192, number of used features: 17
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.365980 -> initscore=-0.549504
[LightGBM] [Info] Start training from score -0.549504
[CV] END learning_rate=0.01, min_child_samples=50, n_estimators=800, num_leaves=64; total time=   5.4s
[LightGBM] [Info] Number of positive: 22761, number of negative: 39431
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003144 seconds.
You can set `force_col_wise=true` to remove t

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LGBMClassifie...ndom_state=10)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'learning_rate': [0.01, 0.03, ...], 'min_child_samples': [20, 50, ...], 'n_estimators': [300, 500, ...], 'num_leaves': [31, 64, ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used her

In [40]:
best_clf = clf_search.best_estimator_

In [41]:
clf_search.best_params_

{'num_leaves': 31,
 'n_estimators': 300,
 'min_child_samples': 50,
 'learning_rate': 0.03}

In [42]:
reg = LGBMRegressor(objective="mae", random_state=11, n_jobs=-1)

In [43]:
param_grid_reg = {
    "n_estimators": [500, 800, 1200],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "num_leaves": [31, 64],
    "tweedie_variance_power": np.arange(1, 2, 0.2),
}

In [44]:
reg_search = RandomizedSearchCV(
    reg,
    param_grid_reg,
    n_iter=10,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=1,
    random_state=25,
)

In [45]:
print("Starting the hyperparameter tuning for the regressor ...")
reg_search.fit(X_train_reg, y_train_reg)

Starting the hyperparameter tuning for the regressor ...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000561 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2466
[LightGBM] [Info] Number of data points in the train set: 22760, number of used features: 17
[LightGBM] [Info] Start training from score 128.000000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000579 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2466
[LightGBM] [Info] Number of data points in the train set: 22761, number of used features: 17
[LightGBM] [Info] Start training from score 129.899994
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000845 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2458
[LightGBM] [Info] Number of data points in the train set: 2

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LGBMRegressor...ndom_state=11)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'learning_rate': [0.01, 0.03, ...], 'n_estimators': [500, 800, ...], 'num_leaves': [31, 64], 'tweedie_variance_power': array([1. , 1....4, 1.6, 1.8])}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validatio

In [46]:
best_reg = reg_search.best_estimator_

In [47]:
reg_search.best_params_

{'tweedie_variance_power': np.float64(1.2),
 'num_leaves': 64,
 'n_estimators': 500,
 'learning_rate': 0.01}

## Combining the results

In [48]:
p_valid = best_clf.predict_proba(X_valid)[:, 1]

In [49]:
amount_valid = best_reg.predict(X_valid)

In [50]:
best_mae = float("inf")
best_alpha = None
best_pred = None
best_spearman = None

In [51]:
for alpha in np.arange(0.5, 2, 0.2):
    y_pred = (p_valid**alpha) * amount_valid

    mae = mean_absolute_error(y_valid, y_pred)
    corr, p_value = spearmanr(y_valid, y_pred)

    print(f"alpha={alpha} | MAE={mae:.2f} | Spearman={corr:.4f}")

    if mae < best_mae:
        best_mae = mae
        best_alpha = alpha
        best_pred = y_pred
        best_spearman = corr

alpha=0.5 | MAE=83.46 | Spearman=0.3765
alpha=0.7 | MAE=77.78 | Spearman=0.3808
alpha=0.8999999999999999 | MAE=73.79 | Spearman=0.3833
alpha=1.0999999999999999 | MAE=70.94 | Spearman=0.3847
alpha=1.2999999999999998 | MAE=68.89 | Spearman=0.3856
alpha=1.4999999999999998 | MAE=67.41 | Spearman=0.3862
alpha=1.6999999999999997 | MAE=66.35 | Spearman=0.3866
alpha=1.8999999999999997 | MAE=65.59 | Spearman=0.3869


In [52]:
print("-" * 30)
print(f"alpha: {best_alpha}")
print(f"Validation MAE: {best_mae:.2f}")
print(f"Validation Spearman Correlation: {best_spearman:.4f}")
print("-" * 30)

------------------------------
alpha: 1.8999999999999997
Validation MAE: 65.59
Validation Spearman Correlation: 0.3869
------------------------------


In [53]:
importance_df = pd.DataFrame(
    {"Feature": X_train.columns, "Importance": best_reg.feature_importances_}  # type: ignore
).sort_values(by="Importance", ascending=False)

print("\nFeature Importance Table:")
print(importance_df)


Feature Importance Table:
                      Feature  Importance
2            sale_revenue_sum        3274
12               recency_days        3271
3           sale_revenue_mean        2928
15     spend_trajectory_ratio        2870
4   sale_discount_applied_sum        2834
16           churn_risk_ratio        2680
10            prod_brand_mode        2477
13          discount_affinity        2452
9          prod_web_only_mean        1467
5            is_returned_mean        1383
8            is_jan_july_mean        1297
7         pack_delay_days_max        1207
14         recent_spend_ratio         940
11          prod_size_nunique         785
1               prod_id_count         773
0             sale_id_nunique         668
6            is_returned_last         194


## Getting the test results

In [54]:
df_test = pd.read_csv("data/test_final.csv")

In [55]:
test_ids = df_test["cust_id"]

In [56]:
X_test = df_test.drop(columns=["cust_id"])

In [57]:
if "prod_brand_mode" in X_test.columns:
    X_test["prod_brand_mode"] = X_test["prod_brand_mode"].astype("category")

In [58]:
p_test = best_clf.predict_proba(X_test)[:, 1] # type: ignore

In [59]:
amount_test = best_reg.predict(X_test)
amount_test = np.clip(amount_test, 0, None)

In [60]:
final_test_pred = (p_test ** best_alpha) * amount_test

In [61]:
submission = pd.DataFrame({
    "cust_id": test_ids,
    "revenue": final_test_pred
})

In [62]:
submission.head()

,cust_id,revenue
0,222wlefm7esnsi3h,27.552365
1,224j2bblkxseyuux,17.657937
2,224myrd7nlqdkm7j,5.005219
3,224qcoczcvqlu56j,54.276080
4,224rajycdiuglkxy,16.723767


In [63]:
submission.to_csv("results/two_hurdle_predictions_1.csv", index=False)